## Create SYNTHETIC dataset

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Define the number of rows for the synthetic dataset
num_rows = 1000

# 1. hospital_id (Integer: Unique ID, e.g., 101=KorleBu, 102=37Mil, etc.)
hospital_ids = np.random.choice([101, 102, 103, 104], num_rows)

# 2. timestamp (DateTime: Date and Hour of the record)
start_date = datetime(2023, 1, 1)
timestamps = [start_date + timedelta(hours=i) for i in range(num_rows)]

# 3. current_occupancy (Float: Current % of beds filled (0.0 to 1.0))
current_occupancy = np.random.uniform(0.3, 0.95, num_rows)

# 4. staff_on_duty (Integer: Number of active staff in emergency ward)
staff_on_duty = np.random.randint(5, 21, num_rows) # Between 5 and 20 staff

# 5. avg_discharge_rate (Float: Historical discharge rate for this hour)
avg_discharge_rate = np.random.uniform(0.1, 0.6, num_rows)

# 6. emergency_incoming (Integer: Known ambulances currently in transit)
emergency_incoming = np.random.randint(0, 11, num_rows) # Between 0 and 10 ambulances

# 7. target_prob (Float: The AI Target: Prob of available bed)
# This is often derived from other features in a real model. For synthetic data, we'll make it somewhat related
# For simplicity, let's make it inversely related to current_occupancy and emergency_incoming
target_prob = 1 - (current_occupancy * 0.7 + emergency_incoming / 20 * 0.3) + np.random.uniform(-0.1, 0.1, num_rows)
target_prob = np.clip(target_prob, 0.0, 1.0) # Ensure it stays between 0 and 1

# Create the DataFrame
synthetic_df = pd.DataFrame({
    'hospital_id': hospital_ids,
    'timestamp': timestamps,
    'current_occupancy': current_occupancy,
    'staff_on_duty': staff_on_duty,
    'avg_discharge_rate': avg_discharge_rate,
    'emergency_incoming': emergency_incoming,
    'target_prob': target_prob
})

# Display the first 5 rows and information about the DataFrame
print("Synthetic Dataset Head:")
display(synthetic_df.head())

print("\nSynthetic Dataset Info:")
synthetic_df.info()

## LOAD DATASET

In [ ]:
import pandas as pd

# Load the synthetic_data.csv into a DataFrame
df_synthetic = pd.read_csv('/content/drive/MyDrive/synthetic_data.csv')

# Display the first few rows of the DataFrame
print("First 5 rows of synthetic_data.csv:")
display(df_synthetic.head())

First 5 rows of synthetic_data.csv:


,hospital_id,timestamp,current_occupancy,staff_on_duty,avg_discharge_rate,emergency_incoming,target_prob
0,103,2023-01-01 00:00:00,0.850882,19,0.264912,7,0.329115
1,104,2023-01-01 01:00:00,0.799595,5,0.106747,8,0.384789
2,104,2023-01-01 02:00:00,0.484708,11,0.158819,0,0.695587
3,102,2023-01-01 03:00:00,0.908261,17,0.277915,1,0.336958
4,102,2023-01-01 04:00:00,0.440325,13,0.456362,6,0.514298


## Descriptive statsistics

In [ ]:
# Display information about the DataFrame, including data types and non-null values
print("\nDataFrame Info:")
df_synthetic.info()

# Display summary statistics for numerical columns
print("\nSummary Statistics:")
display(df_synthetic.describe())


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   hospital_id         1000 non-null   int64  
 1   timestamp           1000 non-null   object 
 2   current_occupancy   1000 non-null   float64
 3   staff_on_duty       1000 non-null   int64  
 4   avg_discharge_rate  1000 non-null   float64
 5   emergency_incoming  1000 non-null   int64  
 6   target_prob         1000 non-null   float64
dtypes: float64(3), int64(3), object(1)
memory usage: 54.8+ KB

Summary Statistics:


,hospital_id,current_occupancy,staff_on_duty,avg_discharge_rate,emergency_incoming,target_prob
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000
mean,102.528000,0.623663,12.71400,0.351702,5.008000,0.486798
std,1.114656,0.182840,4.60841,0.141914,3.081141,0.150114
min,101.000000,0.300290,5.00000,0.100037,0.000000,0.107599
25%,102.000000,0.467272,9.00000,0.231607,2.000000,0.379606
50%,103.000000,0.629152,13.00000,0.356874,5.000000,0.484999
75%,104.000000,0.776031,17.00000,0.472338,8.000000,0.590985
max,104.000000,0.948749,20.00000,0.599653,10.000000,0.858706


## converting the 'timestamp' column to proper datetime objects

In [ ]:
df_synthetic['timestamp'] = pd.to_datetime(df_synthetic['timestamp'])

# Verify the conversion
print(f"New 'timestamp' dtype: {df_synthetic['timestamp'].dtype}")
display(df_synthetic.head())

New 'timestamp' dtype: datetime64[ns]


,hospital_id,timestamp,current_occupancy,staff_on_duty,avg_discharge_rate,emergency_incoming,target_prob
0,103,2023-01-01 00:00:00,0.850882,19,0.264912,7,0.329115
1,104,2023-01-01 01:00:00,0.799595,5,0.106747,8,0.384789
2,104,2023-01-01 02:00:00,0.484708,11,0.158819,0,0.695587
3,102,2023-01-01 03:00:00,0.908261,17,0.277915,1,0.336958
4,102,2023-01-01 04:00:00,0.440325,13,0.456362,6,0.514298


### Feature Engineering


In [ ]:
# Extract time-based features
df_synthetic['hour'] = df_synthetic['timestamp'].dt.hour
df_synthetic['day_of_week'] = df_synthetic['timestamp'].dt.dayofweek

# One-hot encode hospital_id since it's a categorical identifier
df_synthetic = pd.get_dummies(df_synthetic, columns=['hospital_id'], prefix='hosp')

# Display the new features
display(df_synthetic.head())

,timestamp,current_occupancy,staff_on_duty,avg_discharge_rate,emergency_incoming,target_prob,hour,day_of_week,hosp_101,hosp_102,hosp_103,hosp_104
0,2023-01-01 00:00:00,0.850882,19,0.264912,7,0.329115,0,6,False,False,True,False
1,2023-01-01 01:00:00,0.799595,5,0.106747,8,0.384789,1,6,False,False,False,True
2,2023-01-01 02:00:00,0.484708,11,0.158819,0,0.695587,2,6,False,False,False,True
3,2023-01-01 03:00:00,0.908261,17,0.277915,1,0.336958,3,6,False,True,False,False
4,2023-01-01 04:00:00,0.440325,13,0.456362,6,0.514298,4,6,False,True,False,False


### Feature and Target Selection


In [ ]:
# Define features (X) and target (y)
# We drop 'timestamp' and 'target_prob' from features
features = [col for col in df_synthetic.columns if col not in ['timestamp', 'target_prob']]
X = df_synthetic[features]
y = df_synthetic['target_prob']

print(f"Selected Features: {features}")
print(f"Target Variable: target_prob")

Selected Features: ['current_occupancy', 'staff_on_duty', 'avg_discharge_rate', 'emergency_incoming', 'hour', 'day_of_week', 'hosp_101', 'hosp_102', 'hosp_103', 'hosp_104']
Target Variable: target_prob


### Model Training
We will split the data into a training set (80%) and a testing set (20%), then train a Random Forest Regressor.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Squared Error: 0.0044
R2 Score: 0.7969


 Streamlit Dashboard Development


In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.6 MB/s eta 0:00:00


 creating the `app.py` file. This file contains the logic for the Streamlit dashboard, including inputs for hospital data and a section to display the AI model's prediction.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import os
import pydeck as pdk # Import pydeck

# Force Streamlit to show any startup errors
st.set_page_config(page_title="MediMap Ghana", page_icon="🏥")

# --- MOCK SECURITY SESSION ---
# In a production environment, this would come from a secure OAuth/Auth0 login
if 'staff_id' not in st.session_state:
    st.session_state['staff_id'] = "ST-GH-2024-001"

st.write(f"### 🛠 MediMap System Status: Active | User: {st.session_state['staff_id']}")

try:
    st.title("🏥 MediMap Ghana: AI Bed Management")
    st.markdown("--- ")

    # Verify Model File
    model_path = 'medimap_model.pkl'
    if not os.path.exists(model_path):
        st.error(f"❌ ERROR: Model file '{model_path}' not found!")
        st.stop()

    # Load Model
    model = joblib.load(model_path)

    # Sidebar inputs
    st.sidebar.header("Hospital Input Panel")
    hospital = st.sidebar.selectbox("Target Hospital", ["Korle Bu", "37 Military", "Ridge", "Kasoa General"])
    occupancy = st.sidebar.slider("Current Occupancy (%)\n (0=Empty, 100=Full)", 0, 100, 75) / 100.0
    staff = st.sidebar.number_input("Staff on Duty", 1, 100, 20)
    discharge = st.sidebar.slider("Avg Discharge Rate", 0.0, 1.0, 0.3)
    emergency = st.sidebar.number_input("Emergency Incoming", 0, 50, 5)

    # Features Preparation
    now = datetime.now()
    hosp_map = {"Korle Bu": 101, "37 Military": 102, "Ridge": 103, "Kasoa General": 104}
    h_id = hosp_map[hospital]

    input_row = pd.DataFrame([{
        'current_occupancy': occupancy,
        'staff_on_duty': staff,
        'avg_discharge_rate': discharge,
        'emergency_incoming': emergency,
        'hour': now.hour,
        'day_of_week': now.weekday(),
        'hosp_101': 1 if h_id == 101 else 0,
        'hosp_102': 1 if h_id == 102 else 0,
        'hosp_103': 1 if h_id == 103 else 0,
        'hosp_104': 1 if h_id == 104 else 0
    }])

    cols = ['current_occupancy', 'staff_on_duty', 'avg_discharge_rate', 'emergency_incoming', 'hour', 'day_of_week', 'hosp_101', 'hosp_102', 'hosp_103', 'hosp_104']
    input_row = input_row[cols]

    # Run AI Prediction
    prob = model.predict(input_row)[0]

    # --- SECURITY FRAMEWORK: LOGGING REQUIREMENTS ---
    # In production, these logs would be written to a secure database or CloudWatch
    audit_log = {
        "Staff_ID": st.session_state['staff_id'],
        "Timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
        "Change_Log": f"{hospital} Status Update: Occupancy {occupancy*100}%, Staff: {staff}",
        "Device_IP": "Captured (Local Gateway)" # In Streamlit, this is handled by server headers
    }

    # UI Display
    c1, c2 = st.columns(2)
    c1.metric("Bed Availability Chance", f"{prob*100:.1f}%")

    # Define hospital coordinates (updated with real values)
    hospital_coords = {
        "Korle Bu": {"latitude": 5.548, "longitude": -0.222},
        "37 Military": {"latitude": 5.5928, "longitude": -0.1855},
        "Ridge": {"latitude": 5.5616, "longitude": -0.1987},
        "Kasoa General": {"latitude": 5.5350, "longitude": -0.4350}
    }

    # Prepare data for map (only the selected hospital)
    selected_hospital_name = hospital
    selected_coords = hospital_coords.get(selected_hospital_name)

    if selected_coords:
        map_data = pd.DataFrame([
            {
                'latitude': selected_coords['latitude'],
                'longitude': selected_coords['longitude'],
                'name': selected_hospital_name
            }
        ])

        # Assign color based on probability and display status message
        if prob > 0.6:
            st.success("✅ High probability of bed availability.")
            map_data['color'] = [[0, 255, 0, 160]] # Green (RGBA)
        elif prob > 0.3:
            st.warning("⚠️ Limited capacity. Monitor closely.")
            map_data['color'] = [[255, 255, 0, 160]] # Yellow (RGBA)
        else:
            st.error("🚨 CRITICAL: No beds predicted. Initiate rerouting protocol.")
            map_data['color'] = [[255, 0, 0, 160]] # Red (RGBA)

        st.markdown("### Smart Route (Real-time GIS)")

        # Create a PyDeck layer for the scatterplot
        layer = pdk.Layer(
            "ScatterplotLayer",
            map_data,
            get_position='[longitude, latitude]',
            get_color='color',
            get_radius=200, # Radius in meters
            pickable=True,
            tooltip={
                "html": "<b>Hospital:</b> {name}<br>",
                "style": {"backgroundColor": "steelblue", "color": "white"}
            }
        )

        # Set the initial view state for the map
        view_state = pdk.ViewState(
            latitude=selected_coords['latitude'],
            longitude=selected_coords['longitude'],
            zoom=12,
            pitch=50,
        )

        # Create a PyDeck object and display it
        st.pydeck_chart(pdk.Deck(
            map_style="mapbox://styles/mapbox/light-v9", # Light map style
            initial_view_state=view_state,
            layers=[layer],
        ))

    else:
        st.error(f"Coordinates not found for {selected_hospital_name}")


    # Display Security Audit for verification
    with st.expander("🛡️ Security Framework Audit Log"):
        st.json(audit_log)


except Exception as e:
    st.error(f"Application Error: {str(e)}")

Overwriting app.py


In [ ]:
import joblib
# Save the model to a file
joblib.dump(model, 'medimap_model.pkl')
print("Model saved as medimap_model.pkl")

### Launching the Dashboard
 using `localtunnel` to create a public URL for the Streamlit app.

In [ ]:
# Install localtunnel
!npm install -g localtunnel -q

# Run streamlit in the background
import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Get the public URL
print("Wait a few seconds, then click the link below.")
!npx localtunnel --port 8501

In [ ]:
import joblib

# Save the model to a file
joblib.dump(model, 'medimap_model.pkl')
print("Model saved as medimap_model.pkl")

Model saved as medimap_model.pkl


Launching the Dashboard


In [ ]:
# Install localtunnel
!npm install -g localtunnel -q

# Run streamlit in the background
import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Get the public URL
print("Wait a few seconds, then click the link below.")
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 4s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋Wait a few seconds, then click the link below.
⠙⠹⠸⠼⠴⠦your url is: https://mean-laws-rush.loca.lt
^C


In [ ]:
import time
import subprocess

# 1. Kill any existing streamlit processes to avoid port conflicts
!pkill streamlit

# 2. Run streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# 3. Give it a few seconds to initialize
time.sleep(5)

# 4. Get the IP address (this is often the password required by localtunnel)
print("Your Tunnel Password (IP) is:")
!curl ipv4.icanhazip.com

# 5. Restart localtunnel
print("\nClick the link below and enter the IP address above if prompted:")
!npx localtunnel --port 8501

Your Tunnel Password (IP) is:
34.123.25.200

Click the link below and enter the IP address above if prompted:
⠙⠹⠸⠼⠴your url is: https://smooth-toys-hammer.loca.lt
^C


### Preparing for Hugging Face Deployment
Hugging Face Spaces requires a `requirements.txt` file to know which libraries to install.

In [ ]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn==1.6.1
joblib

Overwriting requirements.txt


Hugging face link
https://halima22-medimap.hf.space